# 5 · The lens: what is the operator conditioned on?

Companion to the tutorial *From Hand-Crafted to LLM-Based Variation Operators in Metaheuristics*. It runs offline: the model is a fixed pool of completions, so every number below comes out the same on your machine.

The two demos so far differ in what the model was told and in what survived the
call. Those are the two descriptors of the tutorial, and this notebook makes them
concrete on a single task.

Nothing here is a claim about which channel performs better. The lens describes a
design space; the tutorial is explicit that no law links richer conditioning to
better optimization.

In [1]:
import _bootstrap
import viz

## One operator, three things to condition it on

Take a single job: *propose the next construction heuristic for TSP*. Three
prompts, same job, taken from the tutorial's own figure.

In [2]:
NUMERIC = '''h1: 427
h2: 411
h3: 419'''

SYMBOLIC = '''def step(T, U):
    c = nearest(T[-1], U)
    return T + [c]'''

LINGUISTIC = ('"NN assigns tours to clusters; '
              'try a savings merge; then apply 2-opt repair"')

for name, block, what in ((' Numeric ', NUMERIC, 'a score trajectory over heuristics'),
                          (' Symbolic ', SYMBOLIC, 'an executable construction rule'),
                          (' Linguistic ', LINGUISTIC, 'a strategy-level instruction')):
    print(name.center(56, '─'))
    print(block)
    print(f'  → {what}\n')

─────────────────────── Numeric ────────────────────────
h1: 427
h2: 411
h3: 419
  → a score trajectory over heuristics

─────────────────────── Symbolic ───────────────────────
def step(T, U):
    c = nearest(T[-1], U)
    return T + [c]
  → an executable construction rule

────────────────────── Linguistic ──────────────────────
"NN assigns tours to clusters; try a savings merge; then apply 2-opt repair"
  → a strategy-level instruction



Each carries a different kind of meaning about the same search.

**Numeric** answers *how good is this*. Encodings, features, scalar scores, ranked
parents, trajectories. It is the base channel because it is what metaheuristics
already consume.

**Symbolic** answers *what does this compute*. Code, syntax trees, formal rules:
content whose behaviour a parser, compiler or executor can check.

**Linguistic** answers *why does this work and what should change*. Critiques,
diagnoses, design principles, strategy notes.

Symbolic and Linguistic are both more explicit about problem structure than
Numeric, and they are **not** ranked against each other. Code can preserve
structure that prose loses; prose can express an abstraction that the code does
not contain.

## Which channel is doing the work

Most real methods put several channels in the prompt at once, so the question is
which one is load-bearing. The tutorial answers it with two tests, in order.

**Operative, not boilerplate.** A fixed header like `[Task: solve TSP]` is natural
language, and it conditions nothing: it says the same thing on every call. Content
counts only if it changes with the search and steers the next proposal.

**Drop the channel.** Remove one channel, keep a valid prompt, and see which
removal changes the proposals most. That is the test, and notebook 6 runs it.

In [3]:
PROMPT = {
    'header':     '[Task: solve TSP]',                       # boilerplate, same every call
    'scores':     'parent scores: 427, 411, 419',            # numeric, changes each call
    'code':       'def step(T, U): ...',                     # symbolic, changes each call
    'reflection': 'the merge step is where it loses; try 2-opt',  # linguistic, changes
}

for slot, text in PROMPT.items():
    operative = slot != 'header'
    print(f'{slot:11s} {"operative" if operative else "boilerplate":12s} {text}')

header      boilerplate  [Task: solve TSP]
scores      operative    parent scores: 427, 411, 419
code        operative    def step(T, U): ...
reflection  operative    the merge step is where it loses; try 2-opt


## The second descriptor: what survives the call

**Transient.** The artifact is the answer. Notebook 3: one tour, one call, nothing
left over.

**Amortized.** The artifact is a method. Notebook 4: the heuristic keeps working
on new instances with the model switched off.

**Transfer.** The artifact is inspectable and can be re-bound to another problem.
The tutorial marks this one as a frontier: there are existence proofs, not a
practice.

Our two demos sit in two different cells of the map.

In [4]:
OURS = {('Numeric', 'Transient'): 'notebook 3\ntsp_transient',
        ('Symbolic', 'Amortized'): 'notebook 4\nbpp_amortized'}
viz.lens(OURS)

'<svg xmlns="http://www.w3.org/2000/svg" width="660" height="272" viewBox="0 0 660 272" font-family="ui-sans-serif,-apple-system,Segoe UI,Roboto,sans-serif"><rect width="660" height="272" fill="#ffffff"/><text x="20.0" y="22.0" font-size="13" fill="#1f2430" text-anchor="start" font-weight="600">What conditions the operator, and what survives the call</text><text x="193.3" y="42.0" font-size="11" fill="#8a8f9a" text-anchor="middle" font-weight="600">Transient</text><text x="372.0" y="42.0" font-size="11" fill="#8a8f9a" text-anchor="middle" font-weight="600">Amortized</text><text x="550.7" y="42.0" font-size="11" fill="#8a8f9a" text-anchor="middle" font-weight="600">Transfer</text><text x="94.0" y="87.0" font-size="11" fill="#8a8f9a" text-anchor="end" font-weight="600">Linguistic</text><rect x="104.0" y="52" width="172.7" height="56" rx="7" fill="#d8dce3" fill-opacity="0.25" stroke="#d8dce3" stroke-width="1"/><text x="190.3" y="76.0" font-size="11" fill="#1f2430" text-anchor="middle" font-weight="normal"></text><rect x="282.7" y="52" width="172.7" height="56" rx="7" fill="#d8dce3" fill-opacity="0.25" stroke="#d8dce3" stroke-width="1"/><text x="369.0" y="76.0" font-size="11" fill="#1f2430" text-anchor="middle" font-weight="normal"></text><rect x="461.3" y="52" width="172.7" height="56" rx="7" fill="#d8dce3" fill-opacity="0.25" stroke="#d8dce3" stroke-width="1"/><text x="547.7" y="76.0" font-size="11" fill="#1f2430" text-anchor="middle" font-weight="normal"></text><text x="94.0" y="149.0" font-size="11" fill="#8a8f9a" text-anchor="end" font-weight="600">Symbolic</text><rect x="104.0" y="114" width="172.7" height="56" rx="7" fill="#d8dce3" fill-opacity="0.25" stroke="#d8dce3" stroke-width="1"/><text x="190.3" y="138.0" font-size="11" fill="#1f2430" text-anchor="middle" font-weight="normal"></text><rect x="282.7" y="114" width="172.7" height="56" rx="7" fill="#2f6fdb" fill-opacity="0.07" stroke="#d8dce3" stroke-width="1"/><text x="369.0" y="138.0" font-size="11" fill="#1f2430" text-anchor="middle" font-weight="normal">notebook 4</text><text x="369.0" y="152.0" font-size="11" fill="#1f2430" text-anchor="middle" font-weight="normal">bpp_amortized</text><rect x="461.3" y="114" width="172.7" height="56" rx="7" fill="#d8dce3" fill-opacity="0.25" stroke="#d8dce3" stroke-width="1"/><text x="547.7" y="138.0" font-size="11" fill="#1f2430" text-anchor="middle" font-weight="normal"></text><text x="94.0" y="211.0" font-size="11" fill="#8a8f9a" text-anchor="end" font-weight="600">Numeric</text><rect x="104.0" y="176" width="172.7" height="56" rx="7" fill="#2f6fdb" fill-opacity="0.07" stroke="#d8dce3" stroke-width="1"/><text x="190.3" y="200.0" font-size="11" fill="#1f2430" text-anchor="middle" font-weight="normal">notebook 3</text><text x="190.3" y="214.0" font-size="11" fill="#1f2430" text-anchor="middle" font-weight="normal">tsp_transient</text><rect x="282.7" y="176" width="172.7" height="56" rx="7" fill="#d8dce3" fill-opacity="0.25" stroke="#d8dce3" stroke-width="1"/><text x="369.0" y="200.0" font-size="11" fill="#1f2430" text-anchor="middle" font-weight="normal"></text><rect x="461.3" y="176" width="172.7" height="56" rx="7" fill="#d8dce3" fill-opacity="0.25" stroke="#d8dce3" stroke-width="1"/><text x="547.7" y="200.0" font-size="11" fill="#1f2430" text-anchor="middle" font-weight="normal"></text><text x="20.0" y="264.0" font-size="10" fill="#8a8f9a" text-anchor="start" font-weight="normal">an empty cell is a gap in the literature, not an oversight</text></svg>'

## The published methods, on the same map

This is the placement figure of the tutorial. A dagger in the paper marks a
boundary case, where the channel description leaves the representation open.

In [5]:
PUBLISHED = {
    ('Numeric', 'Transient'):    'LMX · LLMOA · EvoLLM\nOPRO† · EvoPrompt†',
    ('Symbolic', 'Amortized'):   'FunSearch · LLaMEA',
    ('Linguistic', 'Amortized'): 'EoH · ReEvo',
    ('Linguistic', 'Transfer'):  'LAPT · HiFo-Prompt',
}
viz.lens(PUBLISHED)

'<svg xmlns="http://www.w3.org/2000/svg" width="660" height="272" viewBox="0 0 660 272" font-family="ui-sans-serif,-apple-system,Segoe UI,Roboto,sans-serif"><rect width="660" height="272" fill="#ffffff"/><text x="20.0" y="22.0" font-size="13" fill="#1f2430" text-anchor="start" font-weight="600">What conditions the operator, and what survives the call</text><text x="193.3" y="42.0" font-size="11" fill="#8a8f9a" text-anchor="middle" font-weight="600">Transient</text><text x="372.0" y="42.0" font-size="11" fill="#8a8f9a" text-anchor="middle" font-weight="600">Amortized</text><text x="550.7" y="42.0" font-size="11" fill="#8a8f9a" text-anchor="middle" font-weight="600">Transfer</text><text x="94.0" y="87.0" font-size="11" fill="#8a8f9a" text-anchor="end" font-weight="600">Linguistic</text><rect x="104.0" y="52" width="172.7" height="56" rx="7" fill="#d8dce3" fill-opacity="0.25" stroke="#d8dce3" stroke-width="1"/><text x="190.3" y="76.0" font-size="11" fill="#1f2430" text-anchor="middle" font-weight="normal"></text><rect x="282.7" y="52" width="172.7" height="56" rx="7" fill="#2f6fdb" fill-opacity="0.07" stroke="#d8dce3" stroke-width="1"/><text x="369.0" y="76.0" font-size="11" fill="#1f2430" text-anchor="middle" font-weight="normal">EoH · ReEvo</text><rect x="461.3" y="52" width="172.7" height="56" rx="7" fill="#2f6fdb" fill-opacity="0.07" stroke="#d8dce3" stroke-width="1"/><text x="547.7" y="76.0" font-size="11" fill="#1f2430" text-anchor="middle" font-weight="normal">LAPT · HiFo-Prompt</text><text x="94.0" y="149.0" font-size="11" fill="#8a8f9a" text-anchor="end" font-weight="600">Symbolic</text><rect x="104.0" y="114" width="172.7" height="56" rx="7" fill="#d8dce3" fill-opacity="0.25" stroke="#d8dce3" stroke-width="1"/><text x="190.3" y="138.0" font-size="11" fill="#1f2430" text-anchor="middle" font-weight="normal"></text><rect x="282.7" y="114" width="172.7" height="56" rx="7" fill="#2f6fdb" fill-opacity="0.07" stroke="#d8dce3" stroke-width="1"/><text x="369.0" y="138.0" font-size="11" fill="#1f2430" text-anchor="middle" font-weight="normal">FunSearch · LLaMEA</text><rect x="461.3" y="114" width="172.7" height="56" rx="7" fill="#d8dce3" fill-opacity="0.25" stroke="#d8dce3" stroke-width="1"/><text x="547.7" y="138.0" font-size="11" fill="#1f2430" text-anchor="middle" font-weight="normal"></text><text x="94.0" y="211.0" font-size="11" fill="#8a8f9a" text-anchor="end" font-weight="600">Numeric</text><rect x="104.0" y="176" width="172.7" height="56" rx="7" fill="#2f6fdb" fill-opacity="0.07" stroke="#d8dce3" stroke-width="1"/><text x="190.3" y="200.0" font-size="11" fill="#1f2430" text-anchor="middle" font-weight="normal">LMX · LLMOA · EvoLLM</text><text x="190.3" y="214.0" font-size="11" fill="#1f2430" text-anchor="middle" font-weight="normal">OPRO† · EvoPrompt†</text><rect x="282.7" y="176" width="172.7" height="56" rx="7" fill="#d8dce3" fill-opacity="0.25" stroke="#d8dce3" stroke-width="1"/><text x="369.0" y="200.0" font-size="11" fill="#1f2430" text-anchor="middle" font-weight="normal"></text><rect x="461.3" y="176" width="172.7" height="56" rx="7" fill="#d8dce3" fill-opacity="0.25" stroke="#d8dce3" stroke-width="1"/><text x="547.7" y="200.0" font-size="11" fill="#1f2430" text-anchor="middle" font-weight="normal"></text><text x="20.0" y="264.0" font-size="10" fill="#8a8f9a" text-anchor="start" font-weight="normal">an empty cell is a gap in the literature, not an oversight</text></svg>'

The empty cells are the point of drawing it. Symbolic conditioning with a
transient artifact, numeric conditioning with an amortized one: those are not
impossible, they are unoccupied. A map whose gaps are visible is worth more than a
taxonomy that only lists what exists.

## What the lens does not do

It does not predict performance. The tutorial says so in a footnote and repeats it
in the conclusion: the descriptors organize the design space, and whether richer
conditioning helps is an open empirical question, one that has to be answered per
method and per task.

Notebook 6 is what asking that question looks like on one task, with an answer
that came back negative.

## Try it

1. Take a method you know and write down its prompt slots. Which of them change
   between calls? That list, not the paper's table, is its channel set.
2. Place it on the map. If you hesitate between two cells, look at the channel
   description: hesitation usually means it under-determines the representation,
   which is exactly where the tutorial's coders disagreed.
3. Pick an empty cell and design the method that would go there.

---

Next: [6 · Putting the rule to the test](06_drop_channel_ablation.ipynb)